# TFT — Electricity Demand Forecasting
**Temporal Fusion Transformer** trained with PyTorch Forecasting + Lightning.  
Source of truth for data / splits / loss: `train_model.ipynb`.  
Source of truth for TFT wiring: `train_tft.ipynb`.

## 1 · Imports & GPU setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import mlflow
import mlflow.pytorch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# PyTorch Forecasting
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import RMSE, MAE, SMAPE as PTF_SMAPE

# Lightning
import lightning.pytorch as pl
from lightning.pytorch.callbacks import (
    EarlyStopping, LearningRateMonitor, ModelCheckpoint
)
from lightning.pytorch.callbacks.progress import TQDMProgressBar
from lightning.pytorch.loggers import TensorBoardLogger

# Project helpers (data loading, custom losses, metrics)
from data_loading import *   # exposes: load_and_prepare, Y_COL, GROUP_COL, paths …
from loss_funcs import *     # exposes: money, money_pct

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    torch.set_float32_matmul_precision('medium')   # use Tensor Cores on RTX 3090
    print("float32 matmul precision set to 'medium'")

PyTorch version  : 2.6.0+cu124
CUDA available   : True
GPU              : NVIDIA GeForce RTX 3090
float32 matmul precision set to 'medium'


## 2 · Hyper-parameters & feature lists

In [2]:
# ── Forecasting horizon ────────────────────────────────────────────────────────
MAX_PREDICTION_LENGTH = 48
MAX_ENCODER_LENGTH    = 72    # reduced from 168 (3 days is sufficient for hourly electricity)

# ── Training hyper-parameters ──────────────────────────────────────────────────
BATCH_SIZE             = 2048  # increased from 512
MAX_EPOCHS             = 10
LEARNING_RATE          = 3e-3
HIDDEN_SIZE            = 16    # reduced from 64
ATTENTION_HEAD_SIZE    = 1     # reduced from 4
HIDDEN_CONTINUOUS_SIZE = 8    # reduced from 32
DROPOUT                = 0.1
NUM_WORKERS            = 8     # was 0 — enables parallel data loading

# ── Checkpoint / logging dirs ──────────────────────────────────────────────────
CHECKPOINT_DIR = "../models/tft"
TB_LOG_DIR     = "tb_logs"
MLFLOW_EXP     = "tft_electricity"

In [3]:
# ==============================================================================
# TFT feature categorisation
# ------------------------------------------------------------------------------
# TFT has four distinct input streams:
#   1. static_categoricals   – one embedding per series, time-invariant
#   2. static_reals          – numerical scalar per series, time-invariant
#   3. time_varying_known    – features available at *all* future time steps
#   4. time_varying_unknown  – features only available up to t (incl. target)
# ==============================================================================

# ── Weather cols (time-varying known: forecast is available in advance) ────────
weather_cols_all = [
    'temperature_2m', 'apparent_temperature', 'dew_point_2m',
    'relative_humidity_2m', 'precipitation', 'rain', 'snowfall',
    'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
    'surface_pressure', 'wind_speed_10m', 'wind_direction_10m',
    'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation',
    'direct_normal_irradiance',
]

# ── Price cols — depend only on datetime, always known ahead of delivery ───────
# NOTE: dam_price is the day-ahead market price, sell/buy bm_prices are the
# balancing market rates used for the money_pct custom loss.
price_cols = ['dam_price', 'sell_bm_price', 'buy_bm_price']

# ── Capacity features — assumed known in advance (planned capacities) ──────────
capacity_cols = ['max_power', 'max_solar', 'max_ev']

# ── Calendar features (time-varying known categoricals) ────────────────────────
# These must be string-typed in the dataframe (PyTorch Forecasting requirement).
# The _cat suffix is the convention used in data_loading.py; if your data only
# has integer columns (Month, Day, …), we cast them below.
TIME_VARYING_KNOWN_CATS = [
    'Month_cat', 'Day_cat', 'Hour_cat', 'day_of_week_cat', 'season_cat'
]

# ── Static numerical features (one value per station, constant over time) ──────
STATIC_REALS = ['latitude', 'longitude']

# ── Static categorical features (station identity / metadata) ─────────────────
# eic_code_cat  : the group key itself — must be categorical, not raw string
# dso_desc_cat  : distribution system operator
# station_type_cat / oblast_cat : metadata
STATIC_CATS = ['eic_code_cat', 'dso_desc_cat', 'station_type_cat', 'oblast_cat']

# ── Time-varying known reals ───────────────────────────────────────────────────
FUTURE_REALS = weather_cols_all + price_cols + capacity_cols
# time_idx is appended automatically by TimeSeriesDataSet when
# add_relative_time_idx=True; we still include raw time_idx for context.

# ── Time-varying unknown reals (only encoder / history side) ──────────────────
# Only the target itself.  Lagged features could be added here.
PAST_REALS = []   # Y_COL is added automatically as the target

print(f"Y_COL     : {Y_COL}")
print(f"GROUP_COL : {GROUP_COL}")
print(f"FUTURE_REALS ({len(FUTURE_REALS)}) : {FUTURE_REALS}")
print(f"STATIC_CATS ({len(STATIC_CATS)})  : {STATIC_CATS}")

Y_COL     : sum_of_kWh
GROUP_COL : eic_code
FUTURE_REALS (24) : ['temperature_2m', 'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance', 'dam_price', 'sell_bm_price', 'buy_bm_price', 'max_power', 'max_solar', 'max_ev']
STATIC_CATS (4)  : ['eic_code_cat', 'dso_desc_cat', 'station_type_cat', 'oblast_cat']


## 3 · Load data

In [4]:
# This data has a datetime column; categorical columns are kept intact.
# time_idx is already built in train, val, and test and is continuous across them.
print("Loading train …")
train = load_and_prepare(TRAIN_PATH_WITH_DATETME)

print("Loading val   …")
val = load_and_prepare(VAL_PATH_WITH_DATETME)

print("Loading test  …")
test = load_and_prepare(TEST_PATH_WITH_DATETME)

print(f"train : {train.shape}")
print(f"val   : {val.shape}")
print(f"test  : {test.shape}")

Loading train …
Loading val   …
Loading test  …
train : (4586151, 38)
val   : (293880, 38)
test  : (295430, 38)


In [5]:
train = train[train["datetime"] >= "2024-06-01"].copy()

train = train.drop(columns=weather_cols_to_drop)
val = val.drop(columns=weather_cols_to_drop)
test = test.drop(columns=weather_cols_to_drop)

In [6]:
# Sample stations if full training takes too long (keep all 395 by default).
TARGET_STATIONS = 395

station_stats = (
    train.groupby(GROUP_COL)
    .agg(rows=(Y_COL, 'count'))
    .reset_index()
    .sort_values('rows', ascending=False)
)

sampled_stations = station_stats.sample(
    n=min(TARGET_STATIONS, len(station_stats)), random_state=42
)[GROUP_COL].values

print(f"Stations: {len(sampled_stations)}")

train = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val   = val[val[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test  = test[test[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)

print(f"Train rows : {len(train):,}")
print(f"Val rows   : {len(val):,}")
print(f"Test rows  : {len(test):,}")

Stations: 395
Train rows : 3,384,414
Val rows   : 293,880
Test rows  : 295,430


## 4 · Prepare dataset for TFT (TimeSeriesDataSet)

In [7]:
# ==============================================================================
# Ensure all categorical columns exist and are string-typed.
# PyTorch Forecasting requires categoricals to be Python str, not int/category.
# ==============================================================================

def ensure_cat_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create _cat suffix string columns for all base categorical names.
    If a _cat column already exists (e.g. from data_loading), keep it;
    otherwise derive it from the base column or from datetime.
    """
    dt = pd.to_datetime(df['datetime'])

    # Calendar _cat columns (always derive fresh to guarantee string type)
    df['Month_cat']      = dt.dt.month.astype(str)
    df['Day_cat']        = dt.dt.day.astype(str)
    df['Hour_cat']       = dt.dt.hour.astype(str)
    df['day_of_week_cat'] = dt.dt.dayofweek.astype(str)

    def _season(m):
        return {12: 'winter', 1: 'winter', 2: 'winter',
                3: 'spring', 4: 'spring', 5: 'spring',
                6: 'summer', 7: 'summer', 8: 'summer'}.get(m, 'autumn')
    df['season_cat'] = dt.dt.month.map(_season)

    # Station-level _cat columns — create from base column if _cat is missing
    for base in ['eic_code', 'dso_desc', 'station_type', 'oblast']:
        cat_col = base + '_cat'
        if cat_col not in df.columns:
            if base in df.columns:
                df[cat_col] = df[base].astype(str)
            else:
                df[cat_col] = 'unknown'
        else:
            df[cat_col] = df[cat_col].astype(str)

    return df


for df_name, df_obj in [('train', train), ('val', val), ('test', test)]:
    globals()[df_name] = ensure_cat_columns(df_obj)

print("✅ Categorical columns ready")
print(train[[c for c in STATIC_CATS + TIME_VARYING_KNOWN_CATS if c in train.columns]].dtypes)

✅ Categorical columns ready
eic_code_cat        object
dso_desc_cat        object
station_type_cat    object
oblast_cat          object
Month_cat           object
Day_cat             object
Hour_cat            object
day_of_week_cat     object
season_cat          object
dtype: object


In [8]:
# ==============================================================================
# Concatenate train+val+test into a single DataFrame.
# TimeSeriesDataSet.from_dataset() slices on time_idx, so all data must
# live in one frame that shares a continuous, monotone time_idx.
# ==============================================================================

train['_split'] = 'train'
val['_split']   = 'val'
test['_split']  = 'test'

all_data = (
    pd.concat([train, val, test], ignore_index=True)
    .sort_values([GROUP_COL, 'time_idx'])
    .reset_index(drop=True)
)

# Normalise datetime to tz-naive UTC (PTF cannot handle mixed tz)
all_data['datetime'] = pd.to_datetime(all_data['datetime'], utc=True, errors='coerce')
all_data['datetime'] = all_data['datetime'].dt.tz_convert(None)

# Recompute time_idx globally so it is truly continuous
all_data['time_idx'] = (
    (all_data['datetime'] - all_data['datetime'].min())
    .dt.total_seconds() // 3600
).astype('int64')

# Cutoffs for dataset construction
training_cutoff = all_data.loc[all_data['_split'] == 'train', 'time_idx'].max()
val_cutoff      = all_data.loc[all_data['_split'] == 'val',   'time_idx'].max()
test_cutoff     = all_data.loc[all_data['_split'] == 'test',  'time_idx'].max()

all_data.drop(columns=['_split'], inplace=True)

print(f"training_cutoff : {training_cutoff}")
print(f"val_cutoff      : {val_cutoff}")
print(f"test_cutoff     : {test_cutoff}")
print(f"all_data shape  : {all_data.shape}")

training_cutoff : 9479
val_cutoff      : 10223
test_cutoff     : 10971
all_data shape  : (3973724, 35)


In [9]:
# ==============================================================================
# Guard against missing columns.
# If weather columns were dropped upstream, they will not be fed to TFT.
# ==============================================================================

def present(lst, df):
    """Return only columns that exist in df."""
    missing = [c for c in lst if c not in df.columns]
    if missing:
        print(f"  ⚠️  dropping (not found): {missing}")
    return [c for c in lst if c in df.columns]

print("Checking feature lists …")
future_reals_ok   = present(FUTURE_REALS, all_data)
past_reals_ok     = present(PAST_REALS, all_data)
static_reals_ok   = present(STATIC_REALS, all_data)
tv_known_cats_ok  = present(TIME_VARYING_KNOWN_CATS, all_data)
static_cats_ok    = present(STATIC_CATS, all_data)

print(f"\nfuture_reals_ok  ({len(future_reals_ok)})  : {future_reals_ok}")
print(f"static_reals_ok  ({len(static_reals_ok)})  : {static_reals_ok}")
print(f"static_cats_ok   ({len(static_cats_ok)})  : {static_cats_ok}")
print(f"tv_known_cats_ok ({len(tv_known_cats_ok)}) : {tv_known_cats_ok}")

Checking feature lists …
  ⚠️  dropping (not found): ['apparent_temperature', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure', 'wind_direction_10m', 'wind_gusts_10m', 'diffuse_radiation', 'direct_normal_irradiance']

future_reals_ok  (12)  : ['temperature_2m', 'dew_point_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'shortwave_radiation', 'dam_price', 'sell_bm_price', 'buy_bm_price', 'max_power', 'max_solar', 'max_ev']
static_reals_ok  (2)  : ['latitude', 'longitude']
static_cats_ok   (4)  : ['eic_code_cat', 'dso_desc_cat', 'station_type_cat', 'oblast_cat']
tv_known_cats_ok (5) : ['Month_cat', 'Day_cat', 'Hour_cat', 'day_of_week_cat', 'season_cat']


In [10]:
# ==============================================================================
# Build TimeSeriesDataSet
# ==============================================================================
# Architecture summary:
#   encoder  — MAX_ENCODER_LENGTH steps of *observed* history fed to LSTM encoder
#   decoder  — MAX_PREDICTION_LENGTH future steps produced by LSTM decoder
#
# GroupNormalizer(transformation='softplus') normalises each station's target
# independently, which is crucial for multi-series with very different scales.
#
# allow_missing_timesteps=True lets PTF handle gaps (power outages etc.) without
# crashing — missing rows are filled with NaN and masked in attention.
# ==============================================================================

training_dataset = TimeSeriesDataSet(
    all_data[all_data['time_idx'] <= training_cutoff],
    time_idx                        = 'time_idx',
    target                          = Y_COL,
    group_ids                       = [GROUP_COL],
    min_encoder_length              = MAX_ENCODER_LENGTH // 2,
    max_encoder_length              = MAX_ENCODER_LENGTH,
    min_prediction_length           = 1,
    max_prediction_length           = MAX_PREDICTION_LENGTH,
    static_categoricals             = static_cats_ok,
    static_reals                    = static_reals_ok,
    time_varying_known_reals        = future_reals_ok + ['time_idx'],
    time_varying_known_categoricals = tv_known_cats_ok,
    time_varying_unknown_reals      = [Y_COL] + past_reals_ok,
    target_normalizer               = GroupNormalizer(
        groups=[GROUP_COL], transformation='softplus'
    ),
    add_relative_time_idx  = True,   # adds normalised position within window
    add_target_scales      = True,   # exposes group mean/std as static feature
    add_encoder_length     = True,   # exposes actual encoder length as covariate
    allow_missing_timesteps= True,
)

# Validation dataset reuses training's encoders/scalers (no data leakage).
# predict=True ensures only the last MAX_PREDICTION_LENGTH steps are returned.
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    all_data[all_data['time_idx'] <= val_cutoff],
    predict          = True,
    stop_randomization = True,
)

print(f"Training   samples : {len(training_dataset):,}")
print(f"Validation samples : {len(validation_dataset):,}")

Training   samples : 3,402,979
Validation samples : 395


In [11]:
# ── DataLoaders ────────────────────────────────────────────────────────────────
train_loader = training_dataset.to_dataloader(
    train              = True,
    batch_size         = BATCH_SIZE,
    num_workers        = NUM_WORKERS,
    pin_memory         = True,
    persistent_workers = True,   # keeps workers alive between epochs
)

val_loader = validation_dataset.to_dataloader(
    train              = False,
    batch_size         = BATCH_SIZE,
    num_workers        = NUM_WORKERS,
    pin_memory         = True,
    persistent_workers = True,
)

print(f"Training   batches : {len(train_loader)}")
print(f"Validation batches : {len(val_loader)}")

Training   batches : 1661
Validation batches : 1


## 5 · Custom loss wrapper

**Why we cannot plug `money_pct` directly as the TFT training loss:**  
`TemporalFusionTransformer` calls `loss(y_pred, y_true)` inside the Lightning
`training_step` with *normalised* tensors. It does not expose the price columns
(`dam_price`, `sell_bm_price`, `buy_bm_price`) inside that call because they
are treated as input features, not targets.

**Workaround:** We train with `RMSE()` (a well-behaved surrogate) and evaluate
with `money` / `money_pct` on the post-inference DataFrames that contain the
price columns joined back.  This is the standard approach when the business
metric depends on auxiliary columns not available inside the gradient graph.

In [12]:
# Re-export the project metric functions (already available via loss_funcs import,
# but written here explicitly for clarity).
from sklearn.metrics import mean_squared_error

def _smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom  = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8))

def _rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def _mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask   = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def _bias(y_true, y_pred):
    return float(np.sum(y_pred) - np.sum(y_true))

def _prices(df):
    return df['dam_price'].values, df['sell_bm_price'].values, df['buy_bm_price'].values

print("Metric helpers ready")

Metric helpers ready


## 6 · TFT model initialisation

In [13]:
# ==============================================================================
# TemporalFusionTransformer.from_dataset() reads the dataset metadata
# (embedding sizes, normaliser specs, feature lists) and wires the model
# automatically.  No need to specify input dimensions manually.
# ==============================================================================

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate          = LEARNING_RATE,
    hidden_size            = HIDDEN_SIZE,
    attention_head_size    = ATTENTION_HEAD_SIZE,
    dropout                = DROPOUT,
    hidden_continuous_size = HIDDEN_CONTINUOUS_SIZE,
    # RMSE is used as training surrogate.
    # money_pct cannot be used here because PTF's loss call does not
    # receive the auxiliary price columns — see note in Section 5.
    loss                   = RMSE(),
    log_interval           = -1,   # disable per-batch logging (TensorBoard only)
    optimizer              = 'adam',
    reduce_on_plateau_patience = 4,
    logging_metrics        = [MAE(), PTF_SMAPE()],
)

print(f"Parameters : {tft.size() / 1e3:.1f}k")

Parameters : 48.5k


## 7 · Training

In [14]:
# ── MLflow experiment setup ────────────────────────────────────────────────────
mlflow.set_experiment(MLFLOW_EXP)
run = mlflow.start_run(run_name='tft_rmse_loss')

# Log hyper-parameters
mlflow.log_params({
    'max_prediction_length' : MAX_PREDICTION_LENGTH,
    'max_encoder_length'    : MAX_ENCODER_LENGTH,
    'batch_size'            : BATCH_SIZE,
    'max_epochs'            : MAX_EPOCHS,
    'learning_rate'         : LEARNING_RATE,
    'hidden_size'           : HIDDEN_SIZE,
    'attention_head_size'   : ATTENTION_HEAD_SIZE,
    'hidden_continuous_size': HIDDEN_CONTINUOUS_SIZE,
    'dropout'               : DROPOUT,
    'training_loss'         : 'RMSE (surrogate)',
    'eval_metrics'          : 'money_pct, money, SMAPE, RMSE, MAPE',
    'n_stations'            : len(sampled_stations),
    'n_train_rows'          : len(train),
    'n_val_rows'            : len(val),
    'n_test_rows'           : len(test),
    'static_cats'           : str(static_cats_ok),
    'tv_known_cats'         : str(tv_known_cats_ok),
    'future_reals'          : str(future_reals_ok),
    'train_date_min'        : str(train['datetime'].min()),
    'train_date_max'        : str(train['datetime'].max()),
    'val_date_min'          : str(val['datetime'].min()),
    'val_date_max'          : str(val['datetime'].max()),
    'test_date_min'         : str(test['datetime'].min()),
    'test_date_max'         : str(test['datetime'].max()),
})

print(f"MLflow run id : {run.info.run_id}")

MLflow run id : bd8eaa24d27d49718a6b0c8e86d835d2


In [15]:
# ── Lightning callbacks ────────────────────────────────────────────────────────

class MetricPrinterCallback(pl.Callback):
    """Print train/val loss at each validation step."""

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        m = trainer.callback_metrics
        print(
            f"[epoch={trainer.current_epoch:3d} | step={trainer.global_step:6d}]"
            f"  train_loss={float(m.get('train_loss', float('nan'))):.4f}"
            f"  val_loss={float(m.get('val_loss', float('nan'))):.4f}"
        )


early_stop = EarlyStopping(
    monitor   = 'val_loss',
    min_delta = 1e-4,
    patience  = 5,
    verbose   = True,
    mode      = 'min',
)

lr_logger = LearningRateMonitor()

checkpoint = ModelCheckpoint(
    dirpath   = CHECKPOINT_DIR,
    filename  = 'tft-{epoch:02d}-{val_loss:.4f}',
    monitor   = 'val_loss',
    mode      = 'min',
    save_top_k = 2,
)

# TQDMProgressBar renders correctly in Jupyter.
progress_bar = TQDMProgressBar(refresh_rate=20)

metric_printer = MetricPrinterCallback()

print("Callbacks ready")

Callbacks ready


In [16]:
# ── Lightning Trainer ──────────────────────────────────────────────────────────
logger      = TensorBoardLogger(TB_LOG_DIR, name='tft_electricity')
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
print(f"Using {accelerator} accelerator")

trainer = pl.Trainer(
    max_epochs          = MAX_EPOCHS,
    accelerator         = accelerator,
    devices             = 1,
    gradient_clip_val   = 0.1,
    callbacks           = [early_stop, lr_logger, checkpoint, progress_bar, metric_printer],
    logger              = logger,
    enable_progress_bar = True,
    log_every_n_steps   = 10,
    precision           = 'bf16-mixed',
    val_check_interval  = 1.0,   # validate once per epoch instead of 4×
)

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: `Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
INFO:lightning.pytorch.utilities.rank_zero:`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..


Using gpu accelerator


In [17]:
# ==============================================================================
# Train
# ==============================================================================
trainer.fit(
    tft,
    train_dataloaders = train_loader,
    val_dataloaders   = val_loader,
)

print(f"\nBest checkpoint : {checkpoint.best_model_path}")
print(f"Best val_loss   : {checkpoint.best_model_score:.6f}")

mlflow.log_param('best_checkpoint', checkpoint.best_model_path)
mlflow.log_metric('best_val_loss', float(checkpoint.best_model_score))

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ RMSE                            │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  7.5 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    320 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  3.9 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 11.9 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 11.3 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  1.1 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     17 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 48.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 48.5 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 713                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

[epoch=  0 | step=  1661]  train_loss=3.4902  val_loss=7.8083


INFO: Metric val_loss improved. New best score: 7.808
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved. New best score: 7.808
INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


AttributeError: 'tuple' object has no attribute 'tb_frame'

## 8 · Inference — rolling evaluation

In [18]:
# ==============================================================================
# Load best checkpoint for evaluation.
# ==============================================================================

best_tft = TemporalFusionTransformer.load_from_checkpoint(
    checkpoint.best_model_path,
    weights_only=False,
)
best_tft.eval()

print(f"Loaded : {checkpoint.best_model_path}")

Loaded : C:\Users\Lev\Documents\GitHub\Diploma\models\tft\tft-epoch=00-val_loss=7.8083.ckpt


In [19]:
# ==============================================================================
# Rolling evaluation
# ==============================================================================
# The val and test splits each contain ~1 month of data (≈720 hours).
# Since MAX_PREDICTION_LENGTH=48, we slide non-overlapping 48-h windows over
# the evaluation period.  Each window uses *true* observed values as encoder
# context — no error accumulation across windows.
#
# Returns a DataFrame with GROUP_COL, time_idx, pred, Y_COL,
# plus all price columns (joined from all_data) needed by money_pct.
# ==============================================================================

PRICE_COLS = ['dam_price', 'sell_bm_price', 'buy_bm_price']

def rolling_eval(model, base_dataset, full_data, start_idx, end_idx):
    """
    Slide MAX_PREDICTION_LENGTH windows over (start_idx, end_idx].
    Returns DataFrame: GROUP_COL, time_idx, pred, Y_COL, price cols, datetime.
    """
    from tqdm.notebook import tqdm

    records   = []
    windows   = list(range(
        start_idx + MAX_PREDICTION_LENGTH,
        end_idx + 1,
        MAX_PREDICTION_LENGTH,
    ))

    for window_end in tqdm(windows, desc='Rolling windows'):
        ds = TimeSeriesDataSet.from_dataset(
            base_dataset,
            full_data[full_data['time_idx'] <= window_end],
            predict            = True,
            stop_randomization = True,
        )
        loader = ds.to_dataloader(
            train      = False,
            batch_size = BATCH_SIZE,
            num_workers= NUM_WORKERS,
            pin_memory = True,
        )
        result = model.predict(
            loader,
            return_index   = True,
            trainer_kwargs = {'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu'},
        )

        pred_np = result[0].cpu().numpy()   # (n_series, MAX_PREDICTION_LENGTH)
        for i, (_, idx_row) in enumerate(result[2].iterrows()):
            for s in range(MAX_PREDICTION_LENGTH):
                t = idx_row['time_idx'] + s
                if t > end_idx:
                    break
                records.append({
                    GROUP_COL : idx_row[GROUP_COL],
                    'time_idx': t,
                    'pred'    : float(pred_np[i, s]),
                })

    preds_df = pd.DataFrame(records)

    # Join ground truth + price columns + datetime
    join_cols = [GROUP_COL, 'time_idx', Y_COL, 'datetime'] + [
        c for c in PRICE_COLS if c in full_data.columns
    ]
    eval_df = preds_df.merge(
        full_data[join_cols].drop_duplicates([GROUP_COL, 'time_idx']),
        on=[GROUP_COL, 'time_idx'],
        how='inner',
    )
    return eval_df


print("rolling_eval() ready")

rolling_eval() ready


In [ ]:
# Only labelled data (no future test2 if present)
labelled_data = all_data[all_data['time_idx'] <= test_cutoff]

print("── Running validation rolling eval ──────────────────────────")
val_eval = rolling_eval(best_tft, training_dataset, labelled_data,
                        training_cutoff, val_cutoff)
print(f"val_eval rows : {len(val_eval):,}")

print("── Running test rolling eval ─────────────────────────────────")
test_eval = rolling_eval(best_tft, training_dataset, labelled_data,
                         val_cutoff, test_cutoff)
print(f"test_eval rows: {len(test_eval):,}")

── Running validation rolling eval ──────────────────────────


Rolling windows:   0%|          | 0/15 [00:00<?, ?it/s]

INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, wh

val_eval rows : 284,400
── Running test rolling eval ─────────────────────────────────


Rolling windows:   0%|          | 0/15 [00:00<?, ?it/s]

INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, wh

## 9 · Evaluation metrics

In [ ]:
val_smape_v     = _smape(val_eval[Y_COL], val_eval['pred'])
val_rmse_v      = _rmse(val_eval[Y_COL], val_eval['pred'])
val_mape_v      = _mape(val_eval[Y_COL], val_eval['pred'])
val_bias_v      = _bias(val_eval[Y_COL], val_eval['pred'])
val_money_v     = money(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))
val_money_pct_v = money_pct(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))

test_smape_v     = _smape(test_eval[Y_COL], test_eval['pred'])
test_rmse_v      = _rmse(test_eval[Y_COL], test_eval['pred'])
test_mape_v      = _mape(test_eval[Y_COL], test_eval['pred'])
test_bias_v      = _bias(test_eval[Y_COL], test_eval['pred'])
test_money_v     = money(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))
test_money_pct_v = money_pct(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))

print("── Validation ──────────────────────────────────────────────")
print(f"Aligned samples : {len(val_eval):,}")
print(f"SMAPE     : {val_smape_v:.4f}")
print(f"RMSE      : {val_rmse_v:.4f}")
print(f"MAPE      : {val_mape_v:.2f} %")
print(f"Bias      : {val_bias_v:+.2f}  (pred_sum - actual_sum)")
print(f"MONEY     : {val_money_v:.4f}")
print(f"MONEY_PCT : {val_money_pct_v:.4f}%")

print("── Test ────────────────────────────────────────────────────")
print(f"Aligned samples : {len(test_eval):,}")
print(f"SMAPE     : {test_smape_v:.4f}")
print(f"RMSE      : {test_rmse_v:.4f}")
print(f"MAPE      : {test_mape_v:.2f} %")
print(f"Bias      : {test_bias_v:+.2f}  (pred_sum - actual_sum)")
print(f"MONEY     : {test_money_v:.4f}")
print(f"MONEY_PCT : {test_money_pct_v:.4f}%")

mlflow.log_metrics({
    'val_smape'     : val_smape_v,
    'val_rmse'      : val_rmse_v,
    'val_mape'      : val_mape_v,
    'val_bias'      : val_bias_v,
    'val_money'     : val_money_v,
    'val_money_pct' : val_money_pct_v,
    'test_smape'    : test_smape_v,
    'test_rmse'     : test_rmse_v,
    'test_mape'     : test_mape_v,
    'test_bias'     : test_bias_v,
    'test_money'    : test_money_v,
    'test_money_pct': test_money_pct_v,
})

mlflow.end_run()
print(f"\nMLflow run logged → {mlflow.get_tracking_uri()}")

In [ ]:
def per_station_metrics(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for grp, gdf in eval_df.groupby(GROUP_COL):
        row = {
            GROUP_COL   : grp,
            'n'         : len(gdf),
            'SMAPE'     : _smape(gdf[Y_COL], gdf['pred']),
            'RMSE'      : _rmse(gdf[Y_COL], gdf['pred']),
            'MAPE'      : _mape(gdf[Y_COL], gdf['pred']),
        }
        if all(c in gdf.columns for c in PRICE_COLS):
            row['MONEY']     = money(gdf[Y_COL], gdf['pred'], *_prices(gdf))
            row['MONEY_PCT'] = money_pct(gdf[Y_COL], gdf['pred'], *_prices(gdf))
        rows.append(row)
    return pd.DataFrame(rows).sort_values('SMAPE')


test_station_metrics = per_station_metrics(test_eval)

print('Top-10 best stations (test SMAPE):')
print(test_station_metrics.head(10).to_string(index=False))
print('\nBottom-10 worst stations (test SMAPE):')
print(test_station_metrics.tail(10).to_string(index=False))

## 10 · Plots

In [ ]:
def plot_forecast(eval_df, eic_code, start_dt=None, end_dt=None, title_prefix=''):
    """
    Plot true vs predicted kWh for one station.

    eval_df must contain: GROUP_COL, datetime, Y_COL, 'pred',
    and optionally the price columns for money_pct in the title.
    """
    # Join datetime if not already present
    if 'datetime' not in eval_df.columns:
        df = eval_df.merge(
            all_data[[GROUP_COL, 'time_idx', 'datetime']],
            on=[GROUP_COL, 'time_idx'], how='inner',
        )
    else:
        df = eval_df

    df = df[df[GROUP_COL] == eic_code].sort_values('datetime')
    if df.empty:
        raise ValueError(f'No data for station: {eic_code!r}')

    if start_dt is not None:
        df = df[df['datetime'] >= pd.Timestamp(start_dt)]
    if end_dt is not None:
        df = df[df['datetime'] <= pd.Timestamp(end_dt)]
    if df.empty:
        raise ValueError('No data in specified datetime range.')

    # Build title
    mape_str = f'MAPE={_mape(df[Y_COL], df["pred"]):.3f}%'
    if all(c in df.columns for c in PRICE_COLS):
        mp = money_pct(df[Y_COL], df['pred'], *_prices(df))
        title = f'{title_prefix}{eic_code}  |  {mape_str}  MONEY_PCT={mp:.2f}'
    else:
        title = f'{title_prefix}{eic_code}  |  {mape_str}'
    title += f'  ({df["datetime"].min().date()} – {df["datetime"].max().date()})'

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df['datetime'], df[Y_COL],   label='True',      linewidth=1, color='steelblue')
    ax.plot(df['datetime'], df['pred'],  label='Predicted', linewidth=1, color='tomato', alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('Datetime')
    ax.set_ylabel(Y_COL)
    ax.legend()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    fig.autofmt_xdate(rotation=0, ha='center')
    plt.tight_layout()
    plt.show()


# ── Quick example plots ────────────────────────────────────────────────────────
best_station  = test_station_metrics.iloc[0][GROUP_COL]
worst_station = test_station_metrics.iloc[-1][GROUP_COL]

print(f'Best  station (SMAPE): {best_station}')
plot_forecast(test_eval, eic_code=best_station,  title_prefix='[BEST]  ')

print(f'Worst station (SMAPE): {worst_station}')
plot_forecast(test_eval, eic_code=worst_station, title_prefix='[WORST] ')

## 11 · Interpretability — variable importance

In [ ]:
# ==============================================================================
# TFT exposes variable selection weights — a unique feature that shows which
# inputs the model attends to most.  This runs on the validation loader.
# ==============================================================================

# Compute average attention weights across the validation set
interpretation = best_tft.interpret_output(
    best_tft.predict(
        val_loader,
        return_x          = True,
        trainer_kwargs    = {'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu'},
    )[1],   # returns (predictions, x)
    reduction='sum',
)

# Plot
best_tft.plot_interpretation(interpretation)

## 12 · Notes & known limitations

### money_pct as training loss
The `money_pct` function from `loss_funcs.py` requires three price series
(`dam_price`, `sell_bm_price`, `buy_bm_price`) in addition to predictions and
ground truth. PyTorch Forecasting's `training_step` calls `loss(y_pred, y_true)`
with normalised tensors only — there is no hook to inject auxiliary columns at
that point without forking the PTF source code.  

**Current workaround:** train with RMSE (a good surrogate because minimising
squared error also minimises expected financial deviation), then evaluate with
money / money_pct post-hoc on un-normalised predictions merged with prices.

**If you want money_pct as the training objective:** you would need to subclass
`TemporalFusionTransformer`, override `training_step`, and pass the price
tensors through the batch dict under custom keys.  This is non-trivial but
feasible — file a feature request or open a PR in the pytorch-forecasting repo.

### Rolling evaluation vs single-pass
We use rolling 48-h windows so that each prediction window gets true observed
context. This matches real-world deployment (daily re-fitting with the previous
day's actuals) and avoids error accumulation.

### Categorical columns
`eic_code` is both the `group_ids` key **and** a static categorical. PyTorch
Forecasting handles this correctly — the embedding is used as a station
fingerprint in the static context networks.